## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [ ]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


데이터 불러오기

In [ ]:
from datasets import load_dataset

dataset = load_dataset("klue/klue", "ynat")

dataset

README.md:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

ynat/train-00000-of-00001.parquet:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

ynat/validation-00000-of-00001.parquet:   0%|          | 0.00/847k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45678 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9107 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})

기본 모델과 토크나이저 불러오기

In [ ]:
label_names = dataset["train"].features["label"].names
print("라벨 목록:", label_names)

model_name = "klue/bert-base"
num_labels = len(label_names)

tokenizer = AutoTokenizer.from_pretrained(model_name)

라벨 목록: ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
model_before = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
before_classifier = pipeline(
    "text-classification",
    model=model_before,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_texts = [
    "정부, 내년도 예산안 국회 제출",
    "손흥민 시즌 20호골 폭발",
    "새로운 스마트폰 출시로 반도체 수요 급증",
    "봄맞이 벚꽃 축제 전국 곳곳에서 개최"
]

for text in test_texts:
    result = before_classifier(text)
    print("입력:", text)
    print(f"파인튜닝 전 결과: {result[0]['label']} ({result[0]['score']*100:.2f}%)")
    print()

입력: 정부, 내년도 예산안 국회 제출
파인튜닝 전 결과: 경제 (18.86%)

입력: 손흥민 시즌 20호골 폭발
파인튜닝 전 결과: 사회 (20.17%)

입력: 새로운 스마트폰 출시로 반도체 수요 급증
파인튜닝 전 결과: 사회 (20.18%)

입력: 봄맞이 벚꽃 축제 전국 곳곳에서 개최
파인튜닝 전 결과: 스포츠 (19.39%)



In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
eval_dataset = dataset["validation"].shuffle(seed=42).select(range(500))

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['guid', 'title', 'label', 'url', 'date'],
    num_rows: 3000
})
Dataset({
    features: ['guid', 'title', 'label', 'url', 'date'],
    num_rows: 500
})


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["title"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
remove_cols_train = [col for col in ["guid", "title", "url", "date"] if col in tokenized_train.column_names]
remove_cols_eval = [col for col in ["guid", "title", "url", "date"] if col in tokenized_eval.column_names]

tokenized_train = tokenized_train.remove_columns(remove_cols_train)
tokenized_eval = tokenized_eval.remove_columns(remove_cols_eval)

tokenized_train[0]

{'label': 3,
 'input_ids': [2,
  29104,
  3956,
  2223,
  2015,
  18556,
  16342,
  15146,
  100,
  1391,
  2437,
  7683,
  8189,
  3,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./ynat_klue_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.610439,0.674628,0.782000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=188, training_loss=0.8802820862607753, metrics={'train_runtime': 27.8735, 'train_samples_per_second': 107.629, 'train_steps_per_second': 6.745, 'total_flos': 49335537600000.0, 'train_loss': 0.8802820862607753, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
0.610439,0.674628,1,0.782000


{'eval_loss': 0.6746283173561096, 'eval_accuracy': 0.782}

In [ ]:
trainer.save_model("./my_korean_news_model")
tokenizer.save_pretrained("./my_korean_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_korean_news_model/tokenizer_config.json',
 './my_korean_news_model/tokenizer.json')

In [ ]:
after_classifier = pipeline(
    "text-classification",
    model="./my_korean_news_model",
    tokenizer="./my_korean_news_model",
    device=0 if torch.cuda.is_available() else -1
)

for text in test_texts:
    result = after_classifier(text)
    print("입력:", text)
    print(f"파인튜닝 후 결과: {result[0]['label']} ({result[0]['score']*100:.2f}%)")
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

입력: 정부, 내년도 예산안 국회 제출
파인튜닝 후 결과: 정치 (75.06%)

입력: 손흥민 시즌 20호골 폭발
파인튜닝 후 결과: 스포츠 (96.05%)

입력: 새로운 스마트폰 출시로 반도체 수요 급증
파인튜닝 후 결과: 경제 (55.53%)

입력: 봄맞이 벚꽃 축제 전국 곳곳에서 개최
파인튜닝 후 결과: 생활문화 (82.59%)

